# 02.2 — Array axis conventions

The single most common source of confusion when porting from MATLAB neural-data code: **which axis means what?** MATLAB uses one convention, PyTorch another, and the production codebase a third (chosen to match what the model actually needs).

This notebook is the canonical reference for the three conventions you'll encounter:

| Convention | Used by | Shape |
|---|---|---|
| MATLAB **SSCTB** | MATLAB Deep Learning Toolbox | `[C, T, A]` per trial + parallelized over `B` |
| PyTorch **(N, C, H, W)** | PyTorch convolutional models | batch-first |
| Codebase **(W, T, A, C)** | this project's `Dataset` and composites | batch-first; sequence on outer axis |

The codebase's choice isn't arbitrary — it's what falls out when you take MATLAB's per-trial `(C, T, A)` and add `W` windows up front for the GRU's sequence axis.

**Prerequisite:** [02.1 numpy vs MATLAB arrays](02.1_numpy_vs_matlab_arrays.ipynb).

## Section 1 — What MATLAB does

MATLAB's Deep Learning Toolbox uses the **`'SSCTB'`** format for image-like neural data:

```
S  = Spatial dimension (rows / height)
S  = Spatial dimension (cols / width)
C  = Channel
T  = Time
B  = Batch
```

The five SSCTB letters only come into play when MATLAB wraps data in a `dlarray` for training. On disk and inside `cgg_loadDataArray`, this pipeline's per-trial data is a plain 3-D numeric array, `[NumChannels, NumSamples, NumProbes]`:

```matlab
[NumChannels, NumSamples, NumProbes] = size(Data);
% e.g. (58, 3001, 6) for one Decision-epoch trial:
%   - 58 channels (per probe)
%   - 3001 samples in time
%   - 6 probes/areas
```

After windowing via `cgg_loadDataArray`:

```matlab
[NumChannels, DataWidth, NumProbes, NumWindows] = size(this_Data)
% (C=58, T=100, A=6, W=59) — MATLAB's per-trial post-windowing shape
```

When MATLAB trains in a parallel pool, it batches multiple trials and `B` becomes the trailing axis.

## Section 2 — The Python concepts you need

### 2.1 — Why batch-first

Almost every PyTorch operation expects **batch as the leading axis** (`dim=0`). The convention is so universal that every `nn.Module` documents shapes as `(N, ...)` where N is the batch size.

The reason: PyTorch's autograd, optimizer, and DataLoader all loop over the batch axis. Having it first means slicing a batch (`x[:8]`) is contiguous in memory and cheap.

So when porting MATLAB code, your first job is to **move the batch axis from trailing to leading**. After that, the question is what the per-trial axes look like.

### 2.2 — The codebase's per-trial shape: `(W, T, A, C)`

For this project:

| Symbol | Meaning | Typical size |
|---|---|---|
| **W** | Windows per trial | ~59 (depends on DataWidth + WindowStride) |
| **T** | Samples per window | 100 (DataWidth) |
| **A** | Areas / probes | 6 |
| **C** | Channels per area | 58 |

A batched tensor is `(B, W, T, A, C)` — five axes total. The W axis is what the GRU/LSTM iterates over as its sequence. The composite's `FlattenPerWindow` collapses `(T, A, C)` into a single feature axis before the encoder, then `UnflattenPerWindow` restores the multi-dim shape after the decoder.

Visualization below makes this concrete:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, (ax_mat, ax_py) = plt.subplots(1, 2, figsize=(12, 5))

def box(ax, x, y, w, h, label, color):
    rect = mpatches.FancyBboxPatch((x - w / 2, y - h / 2), w, h,
        boxstyle="round,pad=0.05", facecolor=color, edgecolor="black", linewidth=1.3)
    ax.add_patch(rect)
    if "\n" in label:
        head, tail = label.split("\n", 1)
        ax.text(x, y + 0.1, head, ha="center", va="center", fontsize=12, fontweight="bold", family="monospace")
        ax.text(x, y - 0.18, tail, ha="center", va="center", fontsize=9, color="#333")
    else:
        ax.text(x, y, label, ha="center", va="center", fontsize=12, fontweight="bold", family="monospace")

# MATLAB side
ax_mat.set_title("MATLAB cgg_loadDataArray\n(C, T, A, W) per trial",
    fontsize=12, fontweight="bold")
ax_mat.text(3, 5.0, "size(this_Data) =", fontsize=11, ha="center", family="monospace")
box(ax_mat, 1.0, 4.2, 1.4, 0.6, "C\n(58 channels)", color="#cce4ff")
box(ax_mat, 2.5, 4.2, 1.4, 0.6, "T\n(100 samples)", color="#ffe4cc")
box(ax_mat, 4.0, 4.2, 1.4, 0.6, "A\n(6 areas)", color="#e6f0d0")
box(ax_mat, 5.5, 4.2, 1.4, 0.6, "W\n(59 windows)", color="#f0d0e6")
ax_mat.text(3.25, 3.3, "axis 0   axis 1   axis 2   axis 3", fontsize=10, ha="center",
    family="monospace", color="#666")

ax_mat.text(3, 2.4, "Adding batches → (C, T, A, W, B)", fontsize=11, ha="center",
    color="#aa0000", style="italic")
ax_mat.text(3, 1.7, "[batch axis is LAST]", fontsize=10, ha="center", color="#aa0000")
ax_mat.set_xlim(-0.2, 6.5); ax_mat.set_ylim(0.5, 6.0); ax_mat.set_aspect("equal"); ax_mat.axis("off")

# Python / codebase side
ax_py.set_title("This codebase's Dataset\n(W, T, A, C) per trial",
    fontsize=12, fontweight="bold")
ax_py.text(3, 5.0, "x.shape =", fontsize=11, ha="center", family="monospace")
box(ax_py, 1.0, 4.2, 1.4, 0.6, "W\n(59 windows)", color="#f0d0e6")
box(ax_py, 2.5, 4.2, 1.4, 0.6, "T\n(100 samples)", color="#ffe4cc")
box(ax_py, 4.0, 4.2, 1.4, 0.6, "A\n(6 areas)", color="#e6f0d0")
box(ax_py, 5.5, 4.2, 1.4, 0.6, "C\n(58 channels)", color="#cce4ff")
ax_py.text(3.25, 3.3, "axis 0   axis 1   axis 2   axis 3", fontsize=10, ha="center",
    family="monospace", color="#666")

ax_py.text(3, 2.4, "Adding batches → (B, W, T, A, C)", fontsize=11, ha="center",
    color="#aa0000", style="italic")
ax_py.text(3, 1.7, "[batch axis is FIRST]", fontsize=10, ha="center", color="#aa0000")
ax_py.set_xlim(-0.2, 6.5); ax_py.set_ylim(0.5, 6.0); ax_py.set_aspect("equal"); ax_py.axis("off")

plt.tight_layout()
plt.show()

print("Same colored boxes = same axis semantics; the layout differs.")
print("Production: np.transpose(matlab_data, (3, 1, 2, 0)) reorders (C, T, A, W) → (W, T, A, C).")

**The codebase's transpose is exact:** look at `_window_trial` in `data/mat_dataset.py` — `np.transpose(windowed, (3, 1, 2, 0))` takes MATLAB's `(C, T, A, W)` per-trial layout and rotates it to `(W, T, A, C)`. The `(3, 1, 2, 0)` is "new axis 0 = old axis 3, new axis 1 = old axis 1, new axis 2 = old axis 2, new axis 3 = old axis 0.

### 2.3 — PyTorch's convention: `(N, C, H, W)` for images

For convolutional image data PyTorch standardizes on `(N, C, H, W)` — batch, channels, height, width.

This codebase doesn't use the image-style `(N, C, H, W)` shape directly. Its conv-based encoders ('Convolutional' / 'Resnet' / 'Multi-Filter') plug into the same composite slot as the GRU: `FlattenPerWindow` first collapses `(B, W, T, A, C)` → `(B, W, T*A*C)` and hands that to whichever encoder is configured. A conv encoder then internally reshapes that flat input back to 5-D `(B, W, T, A, C)` and runs its kernels along `T` within each area (see `ConvolutionalEncoder.forward` in `src/neural_data_decoding/models/conv_encoder.py`). No stage produces a channels-first `(N, C, T)` tensor — that convention matters when you build a raw `nn.Conv1d` yourself, as Section 5's last error entry explains.

In [ ]:
# Visual: the shape transformation pipeline through the composite
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 4))

stages = [
    ("Raw trial\nin .mat", 1.0, "(C, TT, A)\n(58, 3001, 6)", "#dddddd"),
    ("After window+\ntranspose", 3.0, "(W, T, A, C)\n(59, 100, 6, 58)", "#cce4ff"),
    ("Batch axis\nadded by\nDataLoader", 5.0, "(B, W, T, A, C)\n(B, 59, 100, 6, 58)", "#cce4ff"),
    ("FlattenPerWindow\nfor encoder", 7.0, "(B, W, T·A·C)\n(B, 59, 34800)", "#ffe4cc"),
    ("After GRU\nencoder", 9.0, "(B, W, H)\n(B, 59, latent)", "#e6f0d0"),
    ("UnflattenPerWindow\nfor decoder loss", 11.0, "(B, W, T, A, C)\n(B, 59, 100, 6, 58)", "#cce4ff"),
]

for label, x, shape, color in stages:
    box = mpatches.FancyBboxPatch((x - 0.85, 2.3), 1.7, 1.3,
        boxstyle="round,pad=0.05", facecolor=color, edgecolor="black", linewidth=1.2)
    ax.add_patch(box)
    ax.text(x, 3.4, label, ha="center", va="center", fontsize=9, fontweight="bold")
    ax.text(x, 2.7, shape, ha="center", va="center", fontsize=9, family="monospace")

# Arrows
for x1, x2 in [(1.85, 2.15), (3.85, 4.15), (5.85, 6.15), (7.85, 8.15), (9.85, 10.15)]:
    ax.annotate("", xy=(x2, 2.95), xytext=(x1, 2.95),
        arrowprops=dict(arrowstyle="->", color="black", lw=1.3))

ax.set_xlim(0, 12); ax.set_ylim(1.5, 4.5)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("How a trial flows through the variational composite — shape at each stage",
    fontsize=12)
plt.tight_layout()
plt.show()

**Reading the diagram:**

1. **Raw trial** — what `load_mat()` returns from a `Decision_Data_*.mat` file. MATLAB-native `(C, T, A)`.
2. **After window + transpose** — `MatFileTrialDataset.__getitem__` windows then transposes to `(W, T, A, C)`.
3. **Batch axis added** — `DataLoader` stacks trials along a new leading axis. Now `(B, W, T, A, C)`.
4. **FlattenPerWindow** — collapses `(T, A, C)` into one feature axis so the GRU sees `(B, W, T*A*C)`.
5. **After encoder** — `(B, W, H)` where H is the hidden / latent size. The W axis is preserved.
6. **UnflattenPerWindow** — restores the per-window shape for the reconstruction loss to compare against the original `(B, W, T, A, C)`.

The full round-trip lives in `src/neural_data_decoding/models/composite.py` (the `forward` method on `VariationalComposite`).

## Section 3 — The `neural_data_decoding` implementation

Look at the shape-restoration logic in the composite:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/models/composite.py").read_text().splitlines()
# Find the forward method of VariationalComposite
start = next((i for i, line in enumerate(src) if "def forward(self, x: torch.Tensor) -> VariationalOutput" in line), -1)
for i in range(start, min(start + 52, len(src))):
    print(f"{i + 1:3} | {src[i]}")

**Things to spot:**

* `x0 = self.flatten(x0)` — collapses `(B, W, T, A, C)` → `(B, W, T*A*C)`.
* `encoded = self.encoder(x0)` — encoder receives `(B, W, flat_features)`.
* `rec = self.decoder(z)` — decoder output is `(B, W, flat_features)`.
* `reconstruction = self.unflatten(rec)` — restores `(B, W, T, A, C)` to match the input shape.
* The `_match_shape_5d` helper at the bottom crops/pads if a conv-based decoder produces a slightly different T.

## Section 4 — Hands-on exercises

### Exercise 4.1 — port the MATLAB transpose

MATLAB's `permute(this_Data, [4, 2, 3, 1])` reorders `(C, T, A, W) → (W, T, A, C)` (MATLAB 1-indexed). Write the NumPy `np.transpose(...)` call that does the same (0-indexed).

In [ ]:
import numpy as np
# this_Data shape (C, T, A, W) — MATLAB layout
arr = np.zeros((58, 100, 6, 59))
print("MATLAB layout:", arr.shape)
# Your attempt here


In [ ]:
# Reveal:
rotated = np.transpose(arr, (3, 1, 2, 0))    # new[0] = old[3], new[1] = old[1], new[2] = old[2], new[3] = old[0]
print("After transpose (W, T, A, C):", rotated.shape)

### Exercise 4.2 — predict the batched shape

A `MatFileTrialDataset` produces samples of shape `(W=59, T=100, A=6, C=58)`. A `DataLoader` with `batch_size=8` is built from it. What's the batched-tensor shape? What does it become after `FlattenPerWindow`?

In [ ]:
# Your answer here as a comment


In [ ]:
# Reveal:
# Batched: (8, 59, 100, 6, 58)   — DataLoader prepends the batch axis
# Flattened per window: (8, 59, 100*6*58) = (8, 59, 34800)

## Section 5 — Common errors

### `RuntimeError: input.size(-1) must be equal to input_size. Expected 58, got 34800`

The most common shape mistake: the encoder was built with `in_features=C=58` but the composite passed it the flattened `T*A*C=34800`. Either build the encoder with the flat product or skip the FlattenPerWindow step. (Caught by the smoke run for D.7 / D.8.)

### `RuntimeError: shape '[...]' is invalid for input of size N`

Your `reshape(...)` arguments don't multiply to the actual tensor size. Print `x.shape` and recompute. Use `-1` for one axis to let PyTorch infer.

### MATLAB's `permute([3 1 2])` ported as `transpose((2, 0, 1))`

Always remember to subtract 1 — MATLAB is 1-indexed, NumPy is 0-indexed. The off-by-one bug here is the single most common one when porting.

### Channels in the wrong place inside a Conv layer

PyTorch's `nn.Conv1d` expects `(N, C, T)` — channels SECOND. If you feed it `(N, T, C)` the convolution applies along the wrong axis and produces garbage that trains but never converges. Fix the layout with `tensor.transpose(1, 2)`; to catch the bug early, print `x.shape` right before the conv — a wrong-axis feed usually shows an implausible channel count (e.g. 100 "channels" when you have 6 areas).

## Section 6 — Further reading

- [PyTorch tensor docs](https://pytorch.org/docs/stable/tensors.html#torch.Tensor). Every tensor method.
- [`torch.permute`](https://pytorch.org/docs/stable/generated/torch.permute.html). The PyTorch equivalent of NumPy's full-permutation `np.transpose`. Careful with names: PyTorch's `torch.transpose(d0, d1)` swaps exactly TWO dims — the full-permutation one is `permute`. Coming from NumPy, this naming trap bites everyone once.
- [`einops`](https://einops.rocks/) — a third-party library that lets you write shape transforms as readable strings: `rearrange(x, 'b w t a c -> b w (t a c)')`. Worth knowing about for complex shape work.
- [`MatFileTrialDataset` source](../../src/neural_data_decoding/data/mat_dataset.py) — see the `_window_trial` function for the canonical transpose.

Next up: **[02.3 loading .mat files](02.3_loading_mat_files.ipynb)**.